In [ ]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [ ]:
# Check GPU status
!nvidia-smi

In [ ]:
# For Qwen3-VL
# %%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install transformers==4.57.1
%pip install --no-deps trl==0.22.2

# Configuration

In [ ]:
SEED = 42

# Model configuration
MODEL_ID = 'alxxtexxr/Qwen3-VL-8B-Thinking-VCB8K-SFT-QLoRA-v20260421190643' # Initial: 'alxxtexxr/Qwen3-VL-8B-Thinking-VCB8K'
RESUME_MODEL_ID = None
RESUME_STEP = None

# Data configuration
SFT_DATA_ID = 'alxxtexxr/coco_captions_1.25k_serialized_for_Qwen3-VL-8B-Thinking-VCB8K'

# Training configuration
MINI_BATCH_SIZE = 8
GRAD_ACC_STEPS = 4
NUM_EPOCHS = 20
WARMUP_STEPS = 5
LR = 5e-9 # Initial: 2e-4

In [ ]:
from datetime import datetime

# Resume training configuration
resume_from_checkpoint = bool(RESUME_MODEL_ID)
if resume_from_checkpoint:
    model_name = RESUME_MODEL_ID
    hub_model_id = RESUME_MODEL_ID
    
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=hub_model_id, local_dir=model_name)
    
    if RESUME_STEP:
        resume_from_checkpoint = f"{hub_model_id}/checkpoint-{RESUME_STEP}"
        # Ensure the checkpoint exists
        assert os.path.exists(resume_from_checkpoint), f"Checkpoint {resume_from_checkpoint} does not exist!"
            
else:
    model_name = MODEL_ID
    hub_model_id = f'{MODEL_ID}-SFT-QLoRA-v{datetime.now().strftime("%Y%m%d%H%M%S")}'
project_name = hub_model_id.split('/')[-1]

print("Resume from checkpoint:", resume_from_checkpoint)
print("Model name:", model_name)
print("Project name:", project_name)
print("Hugging Face model ID:", hub_model_id)

In [ ]:
import os
from unsloth import FastVisionModel
import random
import numpy as np
import torch
import transformers

def set_seed(seed, verbose=True):
    # Set random seed for os
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'  # CUDA deterministic behavior (11.2+)

    # Set random seed for NumPy
    np.random.seed(seed)

    # Set random seed for Torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)           # If using multi-GPU
    # torch.use_deterministic_algorithms(True) # ERROR!
    torch.backends.cudnn.deterministic = True  # Ensures deterministic results
    torch.backends.cudnn.benchmark = False     # Avoids non-deterministic algorithms

    # Set random seed for Transformers
    transformers.set_seed(seed)

    # Set random seed for sklearn and Python's own random module
    random.seed(seed) 
    
    if verbose:
        print("Set random seed:", seed)

set_seed(SEED)

In [ ]:
# Fix TorchDynamo issue
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['UNSLOTH_DISABLE_COMPILE'] = '1'
os.environ['UNSLOTH_DISABLE_FUSED_KERNELS'] = '1'

# Fix Unsloth issue
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ''

# Model

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = resume_from_checkpoint if isinstance(resume_from_checkpoint, str) else model_name,
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16-bit LoRA.
    use_gradient_checkpointing = 'unsloth', # True or 'unsloth' for long context
    # use_gradient_checkpointing = False,
    # dtype = torch.float16, # Force FP16
)

In [ ]:
tokenizer.tokenizer.encode("<vthink><|vtok_7550|><|vtok_974|><|vtok_4167|><|vtok_4384|></vthink><think></think>")

In [ ]:
from peft import PeftModelForCausalLM

if not isinstance(model, PeftModelForCausalLM):
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers     = False, # False if not finetuning vision layers
        finetune_language_layers   = True,  # False if not finetuning language layers
        finetune_attention_modules = True,  # False if not finetuning attention layers
        finetune_mlp_modules       = True,  # False if not finetuning MLP layers

        r = 16,          # The larger, the higher the accuracy, but might overfit
        lora_alpha = 16, # Recommended alpha == r at least
        lora_dropout = 0,
        bias = 'none',
        random_state = SEED,
        use_rslora = False,  # We support rank stabilized LoRA
        loftq_config = None, # And LoftQ
        # target_modules = 'all-linear', # Optional now! Can specify a list if needed
    )
else:   
    print("Model is already a PeftModelForCausalLM, skipping PEFT wrapping.")

# Data

In [ ]:
from datasets import load_dataset

dataset = load_dataset(SFT_DATA_ID)
print(dataset)

In [ ]:
train_set = dataset['train']
val_set = dataset['val']

print("Train set:")
print(train_set)
print("\nValidation set:")
print(val_set)

In [ ]:
VTHINK_START = "<vthink>"
VTHINK_END = "</vthink>"

def convert_to_conversation(sample):
    conversation = [
        { 
            "role": "user",
            "content" : [
                {
                    "type" : "text",  
                    # "text": f"Description: {sample['caption']}\n\nBased on the description, provide the visual representations between {VTHINK_START} and {VTHINK_END}",
                    "text": f"{sample['caption']}\n\nBased on the description above, provide the visual representations between {VTHINK_START} and {VTHINK_END}",
                },
                # {"type" : "image", "image" : sample["image"]},
            ],
        },
        { 
            "role" : "assistant",
            "content" : [
                {"type" : "text",  "text"  : f"<vthink>{sample['image_serialized']}</vthink>"},
            ],
        },
    ]
    return { "messages" : conversation }

train_dataset = [convert_to_conversation(sample) for sample in train_set]
val_dataset = [convert_to_conversation(sample) for sample in val_set]

print("Train dataset sample:")
print(train_dataset[0])
print("\nValidation dataset sample:")
print(val_dataset[0])

In [ ]:
FastVisionModel.for_inference(model) # Enable for inference!

image = None
instruction = val_dataset[0]['messages'][0]['content'][0]['text']

messages = [
    {"role": "user", "content": [
        # {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = False)
outputs = model.generate(**inputs, 
                   streamer = text_streamer, 
                   max_new_tokens = 512,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

# Training

In [ ]:
from transformers import EarlyStoppingCallback

from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
import math

FastVisionModel.for_training(model) # Enable for training!

max_steps = math.ceil(len(train_dataset) / (MINI_BATCH_SIZE * GRAD_ACC_STEPS)) * NUM_EPOCHS
print("Calculated max steps:", max_steps)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    args = SFTConfig(
        # Training arguments
        seed = SEED,
        
        per_device_train_batch_size = MINI_BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACC_STEPS,
        
        # max_steps = 50, # For testing
        max_steps = max_steps,
        # num_train_epochs = NUM_EPOCHS,
        # warmup_ratio=0.05,
        warmup_steps = WARMUP_STEPS,
        learning_rate = LR,
        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        max_grad_norm=1.0,
        weight_decay = 0.01,
        
        # Validation arguments
        eval_strategy='steps',
        eval_steps=20,
        
        # Logging arguments
        logging_strategy='steps',
        logging_steps=10,
        # logging_first_step=True,
        report_to=['tensorboard', 'wandb'],
        
        # Saving arguments
        save_strategy='steps',
        # save_steps=1, # For testing
        save_steps=20,
        # save_total_limit=5, # 1 best + 4 recent checkpoints. Warning: It doesn't work
        
        # With load_best_model_at_end=True, your save_strategy will be ignored and default to eval_strategy.
        # So you will find one checkpoint at the end of each epoch.
        # https://discuss.huggingface.co/t/trainer-not-saving-after-save-steps/5464
        load_best_model_at_end=True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,

        output_dir=project_name,
        hub_model_id=hub_model_id,
        push_to_hub=True,
        hub_strategy='all_checkpoints',
        hub_always_push=True,

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 1024,
    ),
    callbacks = [
        EarlyStoppingCallback(
            early_stopping_patience = 2,
            early_stopping_threshold = 0.001,
        )
    ]
)

In [ ]:
trainer_stats = trainer.train(resume_from_checkpoint=resume_from_checkpoint)

# Inference

In [ ]:
FastVisionModel.for_inference(model) # Enable for inference!

image = None
instruction = val_dataset[0]['messages'][0]['content'][0]['text']

messages = [
    {"role": "user", "content": [
        # {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = False)
outputs = model.generate(**inputs, 
                   streamer = text_streamer, 
                   max_new_tokens = 512,
                   use_cache = True, temperature = 1.5, min_p = 0.1)